# Importar Libreria de procesamiento Analitico

In [1]:
import geopandas as gpd
import pandas as pd
import glob
import json
import dash
from dash import dcc, html
from dash import Dash, dcc, html
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# Crear Archivo shapefile consolidado de la temporada

In [2]:
# Ruta donde están tus shapefiles
ruta = "C:/Inteligencia Agricola/Fertilizacion/FERTILIZACIÓN/Real/*.shp"

# Lista de archivos
archivos = glob.glob(ruta)

# Leer y guardar en lista
lista_gdf = []
for archivo in archivos:
    gdf = gpd.read_file(archivo)
    lista_gdf.append(gdf)

# Unir todos los GeoDataFrames
gdf_avance = gpd.GeoDataFrame(pd.concat(lista_gdf, ignore_index=True))

# (Opcional) Asegurar mismo sistema de coordenadas
gdf_avance = gdf_avance.to_crs(epsg=32616)

# Guardar resultado
gdf_avance.to_file("Operacion_Ferty.shp")

print("Shapefiles unidos correctamente ✅")

Shapefiles unidos correctamente ✅


# Carga de Archivos Shapefile

In [3]:
# Leer shapefile
directorio = "C:/Users/E23072/Python_GIS/fertilizacion/Operacion_Ferty.shp"
gdf = gpd.read_file(directorio)

# ⚠️ Asegurar sistema de coordenadas en metros (IMPORTANTE)
if gdf.crs.is_geographic:
    print("Reproyectando a sistema métrico...")
    gdf = gdf.to_crs(epsg=32616)  # Ajusta si es necesario
    
# Ver columnas disponibles
print(gdf.columns)

Index(['CLIENT', 'FARM', 'FIELD', 'OPERATION', 'PRODUCT', 'DATE', 'GPS_Status',
       'Speed', 'spacing', 'skip_ratio', 'population', 'mult_ratio', 'Area',
       'TasaIdeal', 'Elevation', 'singulatio', 'TasaAplica', 'blockage',
       'autosteer', 'XTE', 'geometry'],
      dtype='object')


# Analitica de Calidad de Fertilización

In [4]:
# 🔧 Convertir columnas a numéricas (corrige el error)
gdf["TasaAplica"] = pd.to_numeric(gdf["TasaAplica"], errors="coerce")
gdf["TasaIdeal"] = pd.to_numeric(gdf["TasaIdeal"], errors="coerce")

# Calcular variación porcentual
gdf["variacion_%"] = ((gdf["TasaAplica"] - gdf["TasaIdeal"]) / gdf["TasaIdeal"]) * 100

# Función de clasificación
def clasificar(variacion):
    if -5 <= variacion <= 5:
        return "Dosis Correcta"
    elif variacion < -5:
        return "Dosis Baja"
    else:
        return "Dosis Alta"

# Aplicar clasificación
gdf["categoria"] = gdf["variacion_%"].apply(clasificar)

# 📐 Calcular área
gdf["area_m2"] = gdf.geometry.area
gdf["area_ha"] = gdf["area_m2"] / 10000

# 📊 Resumen por categoría (conteo)
resumen = gdf["categoria"].value_counts()
print("\nResumen de categorías:")
print(resumen)

# 📊 Resumen por área
resumen_area = gdf.groupby("categoria")["area_ha"].sum().reset_index()

# Calcular porcentaje de área
total_area = resumen_area["area_ha"].sum()
resumen_area["porcentaje_%"] = (resumen_area["area_ha"] / total_area) * 100

print("\nResumen de área por categoría (ha y %):")
print(resumen_area)

# Guardar shapefile con nuevos campos
gdf.to_file("Fertilizado_real.shp")

print("\nArchivo guardado como 'resultado_clasificado.shp'")


Resumen de categorías:
Dosis Correcta    24112
Dosis Baja         5080
Dosis Alta         2024
Name: categoria, dtype: int64

Resumen de área por categoría (ha y %):
        categoria     area_ha  porcentaje_%
0      Dosis Alta    9.417905      3.967593
1      Dosis Baja   35.005410     14.747148
2  Dosis Correcta  192.947401     81.285259


C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\552602086.py:40: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("Fertilizado_real.shp")



Archivo guardado como 'resultado_clasificado.shp'


# Asignar nombre de field_fincas de Maestro de Lotes

In [5]:
# 📂 Cargar capa de lotes
lotes = gpd.read_file("C:/Inteligencia Agricola/Cartografia/Lotes_completo_wgs84.shp")

# 🔧 Asegurar mismo sistema de coordenadas
lotes = lotes.to_crs(gdf.crs)

# 👇 Seleccionar solo los campos necesarios de lotes
lotes = lotes[["Estacion", "Codigo", "Lote", "Zona", "geometry"]]

# 🔗 JOIN ESPACIAL (intersección)
gdf_join = gpd.sjoin(gdf, lotes, how="left", predicate="intersects")

# 🧹 Eliminar columnas innecesarias
gdf_join = gdf_join.drop(
    columns=["index_right","spacing","skip_ratio","population","mult_ratio","singulatio","autosteer","blockage"],
    errors="ignore"
)

# 👇 Seleccionar SOLO los campos que quieres en el shapefile final
campos_finales = [
    "Estacion", "Codigo", "Lote", "Zona",
    "Speed","area_ha","TasaIdeal","TasaAplica","variacion_%","categoria","Elevation",
    "geometry"
]

gdf_join = gdf_join[campos_finales]

# 💾 Guardar resultado final
gdf_join.to_file("Fertilizado_real.shp")

print("Shapefile con join espacial y campos filtrados guardado correctamente")

C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\2076523917.py:29: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_join.to_file("Fertilizado_real.shp")


Shapefile con join espacial y campos filtrados guardado correctamente


# Crear archivo para web - dashboard de visualizacion

In [7]:
# Cargar shapefile
gdf = gpd.read_file("Fertilizado_real.shp")

# Asegurar sistema de coordenadas (WGS84 para web)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(epsg=4326)

# Calcular área (ajusta si ya tienes una columna)
gdf["area_ha"] = gdf.geometry.area

# Inicializar app
app = dash.Dash(__name__, suppress_callback_exceptions=True)

# Dropdown de lotes
lotes = gdf["Codigo"].unique()

app.layout = html.Div([
    html.H1("Dashboard Fertilización"),

    dcc.Dropdown(
        id='lote_filter',
        options=[{"label": l, "value": l} for l in lotes],
        multi=True,
        placeholder="Selecciona lote(s)"
    ),

    dcc.Graph(id="mapa"),
    dcc.Graph(id="barra")
])

# Callback
@app.callback(
    [Output("mapa", "figure"),
     Output("barra", "figure")],
    [Input("lote_filter", "value")]
)
def update_dashboard(lotes_sel):

    # Filtrar
    dff = gdf.copy()
    if lotes_sel:
        dff = dff[dff["lote"].isin(lotes_sel)]

    # MAPA
    color_map = {
        "Dosis_Correcta": "green",
        "Dosis_Baja": "red",
        "Dosis_Alta": "blue"
    }

    fig_map = px.choropleth_mapbox(
        dff,
        geojson=dff.geometry,
        locations=dff.index,
        color="categoria",  # columna con Dosis_*
        color_discrete_map={
        "Dosis_Correcta": "green",
        "Dosis_Baja": "red",
        "Dosis_Alta": "blue"
    },
        mapbox_style="open-street-map",
        zoom=14,
        center={
            "lat": dff.geometry.centroid.y.mean(),
            "lon": dff.geometry.centroid.x.mean()
        },
        opacity=0.7
    )

    # Quitar bordes solo para Dosis_Correcta (truco: ancho 0 general)
    fig_map.update_traces(marker_line_width=0)

    # BARRA
    resumen = dff.groupby("categoria")["area_ha"].sum().reset_index()

    fig_bar = px.bar(
        resumen,
        x="categoria",
        y="area_ha",
        color="categoria",
        color_discrete_map=color_map,
        text_auto=True
    )

    fig_bar.update_layout(
        title="Área por categoría",
        xaxis_title="Categoría",
        yaxis_title="Área (ha)"
    )

    return fig_map, fig_bar


# Run app
if __name__ == "__main__":
    app.run(debug=True)

C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\3793466961.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["area_ha"] = gdf.geometry.area


C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\3793466961.py:64: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  "lat": dff.geometry.centroid.y.mean(),
C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\3793466961.py:65: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  "lon": dff.geometry.centroid.x.mean()
C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\3793466961.py:51: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.choropleth_mapbox(
C:\Users\E23072\AppData\Local\Temp\ipykernel_29012\3793466961.py:64: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. U